<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/Daily_challenge_W5_J3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Étape 1 : Importation des bibliothèques nécessaires

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt

### Étape 2 : Charger l'ensemble de données MNIST
Utiliser `keras.datasets.mnist.load_data()` pour charger les données d'entraînement et de test.
Imprimer les formes des données chargées pour comprendre la structure de l'ensemble de données.

In [ ]:
# Charger l'ensemble de données MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Imprimer les formes des données
print(f"Forme de x_train : {x_train.shape}")
print(f"Forme de y_train : {y_train.shape}")
print(f"Forme de x_test : {x_test.shape}")
print(f"Forme de y_test : {y_test.shape}")

# Afficher un exemple d'image
plt.imshow(x_train[0], cmap='gray')
plt.title(f"Exemple de chiffre : {y_train[0]}")
plt.axis('off')
plt.show()

### Étape 3 : Prétraitement des données pour un réseau neuronal entièrement connecté (Dense)

In [ ]:
# Aplatir les images de 28x28 à 784 pixels
x_train_flat = x_train.reshape(-1, 28 * 28)
x_test_flat = x_test.reshape(-1, 28 * 28)

# Normaliser les valeurs des pixels (échelle de 0 à 1)
x_train_flat = x_train_flat.astype('float32') / 255.0
x_test_flat = x_test_flat.astype('float32') / 255.0

# Encoder les étiquettes cibles en one-hot
y_train_one_hot = keras.utils.to_categorical(y_train, num_classes=10)
y_test_one_hot = keras.utils.to_categorical(y_test, num_classes=10)

print(f"Forme de x_train_flat après aplatissement et normalisation : {x_train_flat.shape}")
print(f"Forme de y_train_one_hot après encodage one-hot : {y_train_one_hot.shape}")

### Étape 4 : Construire et entraîner un réseau neuronal entièrement connecté

In [ ]:
# Créer un modèle Sequential
model_dense = keras.Sequential([
    layers.Input(shape=(784,)), # Couche d'entrée
    layers.Dense(128, activation='relu'), # Couche dense avec activation ReLU
    layers.Dense(64, activation='relu'),  # Deuxième couche dense
    layers.Dense(10, activation='softmax') # Couche de sortie avec activation Softmax pour 10 classes
])

# Compiler le modèle
model_dense.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Afficher le résumé du modèle
model_dense.summary()

# Entraîner le modèle
history_dense = model_dense.fit(
    x_train_flat, y_train_one_hot,
    epochs=10,
    batch_size=32,
    validation_split=0.2 # Utiliser 20% des données d'entraînement pour la validation
)

# Évaluer le modèle sur les données de test
loss_dense, accuracy_dense = model_dense.evaluate(x_test_flat, y_test_one_hot)
print(f"\nPrécision du modèle entièrement connecté sur l'ensemble de test : {accuracy_dense:.4f}")

### Étape 5 : Prétraitement des données pour un réseau neuronal convolutif (CNN)

In [ ]:
# Remodeler les données d'entrée à la forme attendue par une couche Conv2D (batch, height, width, channels)
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn = x_test.reshape(-1, 28, 28, 1)

# Normaliser les valeurs des pixels (échelle de 0 à 1)
x_train_cnn = x_train_cnn.astype('float32') / 255.0
x_test_cnn = x_test_cnn.astype('float32') / 255.0

# Encoder les étiquettes cibles en one-hot (réutiliser y_train_one_hot et y_test_one_hot si déjà fait, sinon refaire)
# Si les labels sont des entiers, on peut les re-encoder pour être sûr.
# Dans ce cas, les labels one-hot sont déjà prêts depuis l'étape précédente.
# y_train_one_hot = keras.utils.to_categorical(y_train, num_classes=10)
# y_test_one_hot = keras.utils.to_categorical(y_test, num_classes=10)

print(f"Forme de x_train_cnn après remodelage et normalisation : {x_train_cnn.shape}")

### Étape 6 : Concevoir et entraîner un réseau neuronal convolutif (CNN)

In [ ]:
# Créer un modèle Sequential pour le CNN
model_cnn = keras.Sequential([
    layers.Input(shape=(28, 28, 1)), # Couche d'entrée
    layers.Conv2D(32, (3, 3), activation='relu'), # Première couche de convolution
    layers.MaxPooling2D((2, 2)), # Couche de pooling
    layers.Conv2D(64, (3, 3), activation='relu'), # Deuxième couche de convolution
    layers.MaxPooling2D((2, 2)), # Deuxième couche de pooling
    layers.Flatten(), # Aplatir la sortie des couches convolutives
    layers.Dense(64, activation='relu'), # Couche dense
    layers.Dense(10, activation='softmax') # Couche de sortie
])

# Compiler le modèle
model_cnn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Afficher le résumé du modèle
model_cnn.summary()

# Entraîner le modèle
history_cnn = model_cnn.fit(
    x_train_cnn, y_train_one_hot,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

# Évaluer le modèle sur les données de test
loss_cnn, accuracy_cnn = model_cnn.evaluate(x_test_cnn, y_test_one_hot)
print(f"\nPrécision du modèle CNN sur l'ensemble de test : {accuracy_cnn:.4f}")

### Étape 7 : Comparer les performances

In [ ]:
print("\n--- Comparaison des performances des modèles ---")
print(f"Précision du modèle entièrement connecté : {accuracy_dense:.4f}")
print(f"Précision du modèle CNN : {accuracy_cnn:.4f}")

# Visualisation de l'historique d'entraînement
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_dense.history['accuracy'], label='Précision d'entraînement (Dense)')
plt.plot(history_dense.history['val_accuracy'], label='Précision de validation (Dense)')
plt.title('Précision du Modèle Entièrement Connecté')
plt.xlabel('Epoch')
plt.ylabel('Précision')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['accuracy'], label='Précision d'entraînement (CNN)')
plt.plot(history_cnn.history['val_accuracy'], label='Précision de validation (CNN)')
plt.title('Précision du Modèle CNN')
plt.xlabel('Epoch')
plt.ylabel('Précision')
plt.legend()

plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_dense.history['loss'], label='Perte d'entraînement (Dense)')
plt.plot(history_dense.history['val_loss'], label='Perte de validation (Dense)')
plt.title('Perte du Modèle Entièrement Connecté')
plt.xlabel('Epoch')
plt.ylabel('Perte')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['loss'], label='Perte d'entraînement (CNN)')
plt.plot(history_cnn.history['val_loss'], label='Perte de validation (CNN)')
plt.title('Perte du Modèle CNN')
plt.xlabel('Epoch')
plt.ylabel('Perte')
plt.legend()

plt.tight_layout()
plt.show()

### Observation :
Comme on peut le constater à partir des précisions affichées et des graphiques, le modèle CNN obtient généralement une meilleure précision sur l'ensemble de test par rapport au modèle entièrement connecté. Cela est dû à la capacité des CNN à capturer des motifs spatiaux dans les images, ce qui est crucial pour les tâches de classification d'images comme MNIST.